# Halstead Table: DEFT vs DEFT_perf




In [5]:
from pathlib import Path

import dill
import pandas as pd
from radon.metrics import h_visit

def _find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'conf').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root from current working directory.')

REPO_ROOT = _find_repo_root()

PATH_DEFT_TREE = REPO_ROOT / 'artifacts/trees/polymerase/deft_lightweight/tree_001_seed42.dill'
PATH_DEFT_PERF_TREE = REPO_ROOT / 'artifacts/trees/polymerase/deft_perf_lightweight/tree_001_seed42.dill'

for p in [PATH_DEFT_TREE, PATH_DEFT_PERF_TREE]:
    if not p.exists():
        raise FileNotFoundError(f'Missing tree file: {p}')

PATH_SAVE_TABLE = REPO_ROOT / 'results/polymerase/halstead_deft_vs_deft_perf.csv'


In [6]:
def load_root(path_tree: Path):
    with open(path_tree, 'rb') as f:
        obj = dill.load(f)
    return obj.root if hasattr(obj, 'root') else obj


def extract_halstead_per_node(root) -> pd.DataFrame:
    rows = []

    def walk(node, label='root'):
        if node is None:
            return

        feature = getattr(node, 'feature', None)
        code = getattr(feature, 'string', None) if feature is not None else None

        if code:
            h = h_visit(code).total
            rows.append({
                'node': label,
                'feature_name': getattr(feature, 'name', 'unknown'),
                'halstead_volume': h.volume,
                'halstead_effort': h.effort,
                'halstead_difficulty': h.difficulty,
            })

        walk(getattr(node, 'left', None), label + '.L')
        walk(getattr(node, 'right', None), label + '.R')

    walk(root)
    return pd.DataFrame(rows)


def aggregate_median(metrics_df: pd.DataFrame) -> dict:
    med = metrics_df[['halstead_volume', 'halstead_effort', 'halstead_difficulty']].median()
    return {
        'halstead_volume': float(med['halstead_volume']),
        'halstead_effort': float(med['halstead_effort']),
        'halstead_difficulty': float(med['halstead_difficulty']),
    }


In [7]:
root_deft = load_root(PATH_DEFT_TREE)
root_perf = load_root(PATH_DEFT_PERF_TREE)

metrics_deft = extract_halstead_per_node(root_deft)
metrics_perf = extract_halstead_per_node(root_perf)

agg_deft = aggregate_median(metrics_deft)
agg_perf = aggregate_median(metrics_perf)

table = pd.DataFrame([
    {
        'Model': 'DEFT',
        'Hal. Vol.': agg_deft['halstead_volume'],
        'Hal. Effort': agg_deft['halstead_effort'],
        'Hal. Dif.': agg_deft['halstead_difficulty'],
    },
    {
        'Model': 'DEFT_perf',
        'Hal. Vol.': agg_perf['halstead_volume'],
        'Hal. Effort': agg_perf['halstead_effort'],
        'Hal. Dif.': agg_perf['halstead_difficulty'],
    },
])

table_display = table.copy()
for c in ['Hal. Vol.', 'Hal. Effort', 'Hal. Dif.']:
    table_display[c] = table_display[c].round(3)

table_display


,Model,Hal. Vol.,Hal. Effort,Hal. Dif.
0,DEFT,15.510,15.510,0.875
1,DEFT_perf,19.755,17.265,1.000
